In [ ]:
from functools import partial
from pathlib import Path
import os
import shutil

from urbanopt_des.uo_cli_wrapper import UOCliWrapper
from tools.docker_uo_cli_wrapper import DockerUOCliWrapper

# ── Execution mode ───────────────────────────────────────────────────────────
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = False
DOCKER_IMAGE_TAG = "urbanopt-cli:1.2-ubuntu22"
# ─────────────────────────────────────────────────────────────────────────────

if USE_DOCKER:
    UO = partial(DockerUOCliWrapper, image_tag=DOCKER_IMAGE_TAG)
else:
    UO = UOCliWrapper

# autoreload the dependencies here when they
# change (specifically the uo_cli_wrapper.py file)
%load_ext autoreload
%autoreload 2

In [ ]:
# Get the working directories
workdir = Path().resolve()
print(f"Working directory: {workdir}")

analysis_dir = workdir / "esbe_2026"
if not analysis_dir.exists():
    analysis_dir.mkdir()

template_data_dir = workdir / "data" / "templates"

print(f"template_data_dir: {template_data_dir}")
print(f"analysis_dir: {analysis_dir}")

# Get the number of usable cores for parallel processing by looking at the system (n-2)
num_usable_cores = os.cpu_count() - 2
print(f"Number of usable cores for parallel processing: {num_usable_cores}")

### Gather the data for the OSA / PAT files


In [ ]:
# read in the feature file
import json

# get the results from the activity 3 folder
uo_coincident = UOCliWrapper(
    analysis_dir / "activity_3", "coincident", template_dir=template_data_dir
)
activity_pat_analysis_dir = analysis_dir / "activity_2_pat"
if not activity_pat_analysis_dir.exists():
    activity_pat_analysis_dir.mkdir()

feature_data = uo_coincident.project_path / "class_project_coincident.json"
run_path = uo_coincident.project_path / "run" / "baseline_scenario"

files_to_copy = []

feature_json = None

with open(feature_data, "r") as f:
    feature_json = json.load(f)

    for feature in feature_json["features"]:
        if feature["properties"]["type"] == "Building":
            feature_id = feature["properties"]["id"]
            feature_name = feature["properties"]["name"]

            # go to the run directory and grab the OSM file
            osm_file = run_path / feature_id / "in.osm"
            new_filename = f"{feature_name}.osm"
            files_to_copy.append(
                {
                    "base_dir": "seeds",
                    "feature_name": feature_name,
                    "file": osm_file,
                    "new_filename": new_filename,
                }
            )

    # copy all the files in the directory
    files = (uo_coincident.project_path / "weather").glob("*")
    for file in files:
        files_to_copy.append(
            {
                "base_dir": "weather",
                "file": file,
                "new_filename": file.name,
            }
        )


print(files_to_copy)
for base_dir in (file["base_dir"] for file in files_to_copy):
    if not (activity_pat_analysis_dir / base_dir).exists():
        (activity_pat_analysis_dir / base_dir).mkdir()

# print the files first
for file in files_to_copy:
    print(
        f"Copying {file['file']} to {activity_pat_analysis_dir / file['base_dir'] / file['new_filename']}"
    )


for file in files_to_copy:
    shutil.copy(
        file["file"],
        activity_pat_analysis_dir / file["base_dir"] / file["new_filename"],
    )


# NOW FOR THE CRAZY PART...

# Run the uo_building_to_osa.rb file to convert the osa data. Then grab the measures from
# that result and place in the activity_2_pat folder